# WaterWallet research visualisations

This notebook creates reproducible, publication-ready figures for the WaterWallet
capstone paper. It uses observed Open-Meteo daily precipitation for Thane when the
service is reachable, validates the returned dates before analysis, and saves all
outputs below `waterwallet_research_visualisations/`.

**Evidence categories.** The rainfall figures and metrics are newly generated from
downloaded observations and the current code configuration. Previous paper metrics
are retained only as comparison values. The optional storage figure is a new,
controlled engineering sensitivity experiment using WaterWallet's implemented
daily water-balance function; it is not an observed field result.

In [ ]:
from __future__ import annotations

import json
import platform
import shutil
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pmdarima
import requests
import sklearn
from pmdarima import auto_arima
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT = Path.cwd()
# Support execution from either the repository root or the output folder.
if not (ROOT / "backend").exists():
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / "waterwallet_research_visualisations"
FIG_DIR = OUTPUT_DIR / "waterwallet_figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

THANE_LAT, THANE_LON = 19.22, 72.97
START_DATE, END_DATE = "2010-01-01", "2025-12-31"
TRAIN_END, TEST_START = pd.Timestamp("2021-12-01"), pd.Timestamp("2022-01-01")
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

plt.style.use("default")
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.titlesize": 11, "axes.labelsize": 10,
    "legend.fontsize": 8, "figure.dpi": 120,
})

print(f"Repository: {ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

## 1. Retrieve and validate observed daily precipitation

The request follows the previous research setup: Open-Meteo Archive API,
Thane (19.22° N, 72.97° E), daily `precipitation_sum`, 2010-01-01 through
2025-12-31, timezone Asia/Kolkata. A locally saved CSV is reused on subsequent
runs. Missing values, duplicate dates, gaps, negative values, and mismatched
date coverage raise errors rather than being silently imputed.

In [ ]:
DAILY_CSV = OUTPUT_DIR / "thane_daily_precipitation_2010_2025.csv"
source_details = {
    "source": "Open-Meteo Archive API",
    "endpoint": "https://archive-api.open-meteo.com/v1/archive",
    "parameters": {
        "latitude": THANE_LAT, "longitude": THANE_LON,
        "start_date": START_DATE, "end_date": END_DATE,
        "daily": "precipitation_sum", "timezone": "Asia/Kolkata",
    },
}

if DAILY_CSV.exists():
    daily = pd.read_csv(DAILY_CSV, parse_dates=["date"])
    source_details["retrieval_mode"] = "reused_local_csv"
else:
    response = requests.get(source_details["endpoint"], params=source_details["parameters"], timeout=60)
    response.raise_for_status()
    payload = response.json()
    daily_block = payload.get("daily", {})
    daily = pd.DataFrame({"date": daily_block.get("time", []), "rain_mm": daily_block.get("precipitation_sum", [])})
    daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
    daily.to_csv(DAILY_CSV, index=False)
    source_details["retrieval_mode"] = "downloaded"
    source_details["retrieved_at_utc"] = datetime.now(timezone.utc).isoformat()

daily["date"] = pd.to_datetime(daily["date"], errors="coerce")
daily["rain_mm"] = pd.to_numeric(daily["rain_mm"], errors="coerce")
expected_dates = pd.date_range(START_DATE, END_DATE, freq="D")
validation = {
    "expected_daily_records": len(expected_dates),
    "actual_daily_records": len(daily),
    "missing_dates": int(len(expected_dates.difference(pd.DatetimeIndex(daily["date"])))),
    "duplicate_dates": int(daily["date"].duplicated().sum()),
    "missing_or_non_numeric_precipitation": int(daily["rain_mm"].isna().sum()),
    "negative_precipitation": int((daily["rain_mm"] < 0).sum()),
}
if (validation["actual_daily_records"] != validation["expected_daily_records"] or validation["missing_dates"]
        or validation["duplicate_dates"] or validation["missing_or_non_numeric_precipitation"]
        or validation["negative_precipitation"]):
    raise ValueError(f"Daily rainfall validation failed: {validation}")

daily = daily.sort_values("date").reset_index(drop=True)
daily["year"] = daily["date"].dt.year
daily["month"] = daily["date"].dt.month
monthly = (daily.groupby(["year", "month"], as_index=False)
           .agg(rain_mm=("rain_mm", "sum"), peak_day_mm=("rain_mm", "max"), rainy_days=("rain_mm", lambda s: int((s > 1).sum()))))
monthly["date"] = pd.to_datetime(monthly["year"].astype(str) + "-" + monthly["month"].astype(str).str.zfill(2) + "-01")
monthly = monthly.sort_values("date").reset_index(drop=True)
expected_months = pd.date_range("2010-01-01", "2025-12-01", freq="MS")
if len(monthly) != len(expected_months) or not monthly["date"].equals(pd.Series(expected_months, name="date")):
    raise ValueError("Monthly aggregation is not a complete chronological 2010-01 to 2025-12 series.")
monthly.to_csv(OUTPUT_DIR / "thane_monthly_precipitation_2010_2025.csv", index=False)
print(json.dumps(validation, indent=2))
print(f"Monthly observations: {len(monthly)}; annual mean: {monthly.groupby('year')['rain_mm'].sum().mean():.1f} mm")

## 2. Historical rainfall seasonality

Monthly totals are shown by calendar month. The distribution is based on sixteen
monthly observations per calendar month, preserving interannual variation rather
than averaging it away.

In [ ]:
month_labels = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
grouped = [monthly.loc[monthly["month"] == m, "rain_mm"].to_numpy() for m in range(1, 13)]
fig, ax = plt.subplots(figsize=(8.6, 4.4), constrained_layout=True)
box = ax.boxplot(grouped, patch_artist=True, widths=0.60, showfliers=True,
                 medianprops={"color": "#222222", "linewidth": 1.4},
                 whiskerprops={"color": "#555555"}, capprops={"color": "#555555"})
for patch in box["boxes"]:
    patch.set(facecolor="#8dbbd8", edgecolor="#2d628c", alpha=0.9)
ax.set_xticks(range(1, 13), month_labels)
ax.set_xlabel("Calendar month")
ax.set_ylabel("Monthly precipitation (mm)")
ax.set_title("Thane monthly precipitation distribution, 2010–2025")
ax.grid(axis="y", color="#d9d9d9", linewidth=0.7)
ax.text(7.5, ax.get_ylim()[1] * 0.93, "Southwest monsoon months", ha="center", va="top", fontsize=8, color="#555555")
fig.savefig(FIG_DIR / "fig1_monthly_rainfall_seasonality.png", dpi=300, bbox_inches="tight")
plt.show()

## 3. Chronological holdout evaluation

The model-development period is January 2010–December 2021 (144 months); the
untouched holdout is January 2022–December 2025 (48 months). The seasonal-naive
baseline uses the observation from the same month one year earlier. The recursive
Random Forest is reproduced from the prior notebook’s documented feature set and
hyperparameters; after January 2022 it appends its own predictions, never future
observed rainfall, to the lag history. Auto-SARIMA uses the current FastAPI
service search bounds (`p,q≤2`, `P,Q≤1`, `max_order=5`) for source-code alignment.

In [ ]:
train = monthly.loc[monthly["date"] <= TRAIN_END, ["date", "year", "month", "rain_mm"]].copy()
test = monthly.loc[monthly["date"] >= TEST_START, ["date", "year", "month", "rain_mm"]].copy()
if len(train) != 144 or len(test) != 48:
    raise ValueError(f"Unexpected split sizes: train={len(train)}, test={len(test)}")
y_true = test["rain_mm"].to_numpy(dtype=float)

lookup = dict(zip(monthly["date"], monthly["rain_mm"]))
baseline = np.array([lookup[row.date - pd.DateOffset(years=1)] for row in test.itertuples()], dtype=float)

train_series = train.set_index("date")["rain_mm"].asfreq("MS")
sarima = auto_arima(
    train_series, seasonal=True, m=12, start_p=0, start_q=0, max_p=2, max_q=2,
    start_P=0, start_Q=0, max_P=1, max_Q=1, max_order=5, stepwise=True,
    information_criterion="aic", with_intercept="auto", maxiter=50,
    suppress_warnings=True, error_action="ignore",
)
sarima_pred = np.maximum(np.asarray(sarima.predict(n_periods=len(test)), dtype=float), 0.0)
sarima_order = {"order": list(sarima.order), "seasonal_order": list(sarima.seasonal_order)}

rf_data = monthly[["date", "year", "month", "rain_mm"]].copy()
rf_data["rain_lag1"] = rf_data["rain_mm"].shift(1)
rf_data["rain_lag12"] = rf_data["rain_mm"].shift(12)
rf_data["rain_roll3"] = rf_data["rain_mm"].shift(1).rolling(3).mean()
rf_data["month_sin"] = np.sin(2 * np.pi * rf_data["month"] / 12)
rf_data["month_cos"] = np.cos(2 * np.pi * rf_data["month"] / 12)
rf_data = rf_data.dropna().reset_index(drop=True)
rf_features = ["rain_lag1", "rain_lag12", "rain_roll3", "month_sin", "month_cos"]
rf = RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=2, random_state=RANDOM_SEED, n_jobs=-1)
rf_train = rf_data.loc[rf_data["year"] <= 2021]
rf.fit(rf_train[rf_features], rf_train["rain_mm"])
history = train[["date", "rain_mm"]].copy()
rf_pred = []
for target in test.itertuples():
    values = history["rain_mm"].to_numpy(dtype=float)
    features = pd.DataFrame([{
        "rain_lag1": values[-1], "rain_lag12": values[-12], "rain_roll3": values[-3:].mean(),
        "month_sin": np.sin(2 * np.pi * target.date.month / 12),
        "month_cos": np.cos(2 * np.pi * target.date.month / 12),
    }])
    prediction = max(0.0, float(rf.predict(features[rf_features])[0]))
    rf_pred.append(prediction)
    history = pd.concat([history, pd.DataFrame({"date": [target.date], "rain_mm": [prediction]})], ignore_index=True)
rf_pred = np.asarray(rf_pred)

holdout = test[["date", "rain_mm"]].rename(columns={"rain_mm": "actual_mm"}).copy()
holdout["same_month_last_year_mm"] = baseline
holdout["autosarima_mm"] = sarima_pred
holdout["recursive_random_forest_mm"] = rf_pred
holdout.to_csv(OUTPUT_DIR / "holdout_actual_and_predictions_2022_2025.csv", index=False)
print(f"Auto-SARIMA selected order: {sarima_order}")

In [ ]:
fig, ax = plt.subplots(figsize=(10.0, 4.8), constrained_layout=True)
ax.plot(holdout["date"], holdout["actual_mm"], color="#1f1f1f", linewidth=1.9, label="Observed")
ax.plot(holdout["date"], holdout["autosarima_mm"], color="#166c9c", linewidth=1.35, label="Auto-SARIMA")
ax.plot(holdout["date"], holdout["same_month_last_year_mm"], color="#d77a1f", linewidth=1.2, linestyle="--", label="Same-month-last-year")
ax.plot(holdout["date"], holdout["recursive_random_forest_mm"], color="#4b8a43", linewidth=1.2, linestyle=":", label="Recursive Random Forest")
ax.set_title("Chronological 48-month holdout: observed and forecast monthly precipitation")
ax.set_xlabel("Month")
ax.set_ylabel("Monthly precipitation (mm)")
ax.grid(axis="y", color="#d9d9d9", linewidth=0.7)
ax.legend(ncol=2, frameon=False, loc="upper right")
ax.set_xlim(holdout["date"].min(), holdout["date"].max())
fig.autofmt_xdate(rotation=0, ha="center")
fig.savefig(FIG_DIR / "fig2_holdout_actual_vs_predicted.png", dpi=300, bbox_inches="tight")
plt.show()

## 4. Model-performance comparison

Metrics are calculated from the just-created predictions and the same 48 holdout
observations. They are not copied from the previous paper. Reference values from
the earlier notebook are added only to the output table for transparent comparison.
Differences are expected because the present analysis uses a newly retrieved data
snapshot and the current, bounded production Auto-SARIMA search rather than the
older notebook’s unconstrained call.

In [ ]:
prediction_columns = {
    "Auto-SARIMA": "autosarima_mm",
    "Same-month-last-year": "same_month_last_year_mm",
    "Recursive Random Forest": "recursive_random_forest_mm",
}
rows = []
for label, column in prediction_columns.items():
    prediction = holdout[column].to_numpy(dtype=float)
    rows.append({
        "model": label,
        "mae_mm": mean_absolute_error(y_true, prediction),
        "rmse_mm": mean_squared_error(y_true, prediction) ** 0.5,
        "r2": r2_score(y_true, prediction),
    })
metrics = pd.DataFrame(rows)
prior = pd.DataFrame([
    {"model": "Auto-SARIMA", "prior_mae_mm": 63.76, "prior_rmse_mm": 108.66, "prior_r2": 0.823},
    {"model": "Same-month-last-year", "prior_mae_mm": 73.50, "prior_rmse_mm": 127.70, "prior_r2": 0.756},
    {"model": "Recursive Random Forest", "prior_mae_mm": 120.26, "prior_rmse_mm": 180.01, "prior_r2": 0.514},
])
metrics = metrics.merge(prior, on="model", how="left")
for metric in ["mae_mm", "rmse_mm", "r2"]:
    metrics[metric] = metrics[metric].round(3)
metrics.to_csv(OUTPUT_DIR / "holdout_model_metrics_2022_2025.csv", index=False)
display(metrics)

panels = [("mae_mm", "MAE (mm)", "lower is better"), ("rmse_mm", "RMSE (mm)", "lower is better"), ("r2", "R²", "higher is better")]
colors = ["#166c9c", "#d77a1f", "#4b8a43"]
fig, axes = plt.subplots(1, 3, figsize=(10.2, 4.25))
labels = metrics["model"].tolist()
for ax, (metric, title, direction) in zip(axes, panels):
    bars = ax.bar(range(len(metrics)), metrics[metric], color=colors, width=0.65)
    ax.axhline(0, color="#555555", linewidth=0.8)
    ax.set_xticks(range(len(metrics)), ["Auto-\nSARIMA", "Seasonal\nnaive", "Recursive\nRF"])
    ax.set_title(title)
    ax.text(0.5, 1.01, direction, transform=ax.transAxes, ha="center", fontsize=8, color="#555555")
    ax.grid(axis="y", color="#dddddd", linewidth=0.7)
    for bar, value in zip(bars, metrics[metric]):
        offset = max(abs(float(value)) * 0.025, 0.02)
        ax.text(bar.get_x() + bar.get_width()/2, value + (offset if value >= 0 else -offset), f"{value:.3g}", ha="center", va="bottom" if value >= 0 else "top", fontsize=8)
fig.suptitle("Forecast accuracy on the identical 2022–2025 holdout", fontsize=12, y=0.995)
fig.tight_layout(rect=(0, 0, 1, 0.91))
fig.savefig(FIG_DIR / "fig3_model_metrics_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

## 5. Separate 12-month Auto-SARIMA forecast

This model is deliberately fit **after** the holdout evaluation, using the full
2010–2025 series. Its 12 future months are not used in holdout scoring. The band
is the model’s 95% prediction interval; it is model-based uncertainty, not a
guaranteed range.

In [ ]:
full_series = monthly.set_index("date")["rain_mm"].asfreq("MS")
future_model = auto_arima(
    full_series, seasonal=True, m=12, start_p=0, start_q=0, max_p=2, max_q=2,
    start_P=0, start_Q=0, max_P=1, max_Q=1, max_order=5, stepwise=True,
    information_criterion="aic", with_intercept="auto", maxiter=50,
    suppress_warnings=True, error_action="ignore",
)
forecast_values, intervals = future_model.predict(n_periods=12, return_conf_int=True, alpha=0.05)
forecast_values = np.maximum(np.asarray(forecast_values, dtype=float), 0.0)
intervals = np.asarray(intervals, dtype=float)
lower = np.minimum(np.maximum(intervals[:, 0], 0.0), forecast_values)
upper = np.maximum(intervals[:, 1], forecast_values)
future_dates = pd.date_range(full_series.index.max() + pd.offsets.MonthBegin(1), periods=12, freq="MS")
forecast = pd.DataFrame({"date": future_dates, "forecast_mm": forecast_values, "lower_95_mm": lower, "upper_95_mm": upper})
forecast.to_csv(OUTPUT_DIR / "autosarima_12_month_forecast.csv", index=False)
future_order = {"order": list(future_model.order), "seasonal_order": list(future_model.seasonal_order)}
print(f"Full-series Auto-SARIMA order: {future_order}")

history_window = full_series.loc["2021-01-01":]
fig, ax = plt.subplots(figsize=(10.0, 4.8), constrained_layout=True)
ax.plot(history_window.index, history_window.values, color="#1f1f1f", linewidth=1.4, label="Observed monthly precipitation")
ax.plot(forecast["date"], forecast["forecast_mm"], color="#166c9c", linewidth=1.7, marker="o", markersize=3, label="12-month forecast")
ax.fill_between(forecast["date"], forecast["lower_95_mm"], forecast["upper_95_mm"], color="#166c9c", alpha=0.18, label="95% prediction interval")
ax.axvline(full_series.index.max(), color="#666666", linewidth=1.0, linestyle="--")
ax.text(full_series.index.max(), ax.get_ylim()[1] * 0.94, "Forecast origin", ha="right", va="top", fontsize=8, color="#555555")
ax.set_title("Auto-SARIMA 12-month monthly precipitation forecast for Thane")
ax.set_xlabel("Month")
ax.set_ylabel("Monthly precipitation (mm)")
ax.grid(axis="y", color="#d9d9d9", linewidth=0.7)
ax.legend(frameon=False, loc="upper left")
fig.autofmt_xdate(rotation=0, ha="center")
fig.savefig(FIG_DIR / "fig4_autosarima_12_month_forecast.png", dpi=300, bbox_inches="tight")
plt.show()

## 6. Optional engineering figure: tank-capacity sensitivity

This supported additional figure runs the production `simulate_tank_year` function,
not a duplicate notebook calculation. It uses observed 2025 daily rainfall with a
documented controlled configuration: 100 m² gross roof (80 m² usable), concrete
runoff coefficient 0.90, 5% first-flush factor, 20 occupants, fixed eligible
demand 65 LPCD (1,300 L/day), and no recharge diversion. It illustrates a model
sensitivity, not a real society measurement or recommended tank size.

In [ ]:
sys.path.insert(0, str(ROOT / "backend"))
from services.tank_sizer import CANDIDATE_CAPACITIES_L, simulate_tank_year

sensitivity_daily = daily.loc[daily["year"] == 2025, "rain_mm"].to_list()
sensitivity_rows = []
for capacity in CANDIDATE_CAPACITIES_L:
    result = simulate_tank_year(
        rainfall_mm=sensitivity_daily, tank_capacity_litres=float(capacity), daily_demand_litres=1300.0,
        usable_roof_area_sqm=80.0, runoff_coefficient=0.90, first_flush_fraction=0.05,
        year=2025, recharge_enabled=False,
    )
    sensitivity_rows.append({
        "capacity_litres": capacity, "useful_water_litres": result.useful_water_litres,
        "lost_overflow_litres": result.lost_overflow_litres, "unmet_demand_litres": result.unmet_demand_litres,
        "reliability_percent": result.reliability_percent,
    })
sensitivity = pd.DataFrame(sensitivity_rows)
sensitivity.to_csv(OUTPUT_DIR / "tank_capacity_sensitivity_2025_controlled.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.9), constrained_layout=True)
axes[0].plot(sensitivity["capacity_litres"], sensitivity["useful_water_litres"] / 1000, marker="o", color="#166c9c", label="Useful water")
axes[0].plot(sensitivity["capacity_litres"], sensitivity["lost_overflow_litres"] / 1000, marker="s", color="#c85732", label="Lost overflow")
axes[0].set_xlabel("Tank capacity (L)"); axes[0].set_ylabel("Annual volume (kL)")
axes[0].set_title("2025 balance outputs"); axes[0].grid(axis="y", color="#dddddd", linewidth=0.7); axes[0].legend(frameon=False)
axes[1].plot(sensitivity["capacity_litres"], sensitivity["reliability_percent"], marker="o", color="#4b8a43")
axes[1].set_xlabel("Tank capacity (L)"); axes[1].set_ylabel("Eligible-demand reliability (%)")
axes[1].set_title("Reliability under controlled assumptions"); axes[1].grid(axis="y", color="#dddddd", linewidth=0.7)
fig.suptitle("Storage capacity sensitivity using WaterWallet's daily water balance", fontsize=12)
fig.savefig(FIG_DIR / "fig5_tank_capacity_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

## 7. Reproducibility record and output validation

The final cell records data/model settings and verifies that figures are non-empty
300-DPI raster outputs. The ZIP contains the notebook, figures, tables, metadata,
and README—without virtual environments, caches, or unrelated project files.

In [ ]:
required_figures = [
    "fig1_monthly_rainfall_seasonality.png", "fig2_holdout_actual_vs_predicted.png",
    "fig3_model_metrics_comparison.png", "fig4_autosarima_12_month_forecast.png",
    "fig5_tank_capacity_sensitivity.png",
]
figure_validation = {}
for name in required_figures:
    path = FIG_DIR / name
    if not path.exists() or path.stat().st_size < 10_000:
        raise RuntimeError(f"Figure missing or unexpectedly small: {path}")
    image = matplotlib.image.imread(path)
    figure_validation[name] = {"bytes": path.stat().st_size, "height_px": int(image.shape[0]), "width_px": int(image.shape[1])}

metadata = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "data_source": source_details,
    "validation": validation,
    "monthly_observations": int(len(monthly)),
    "train_period": "2010-01 through 2021-12 (144 months)",
    "holdout_period": "2022-01 through 2025-12 (48 months)",
    "autosarima_holdout": sarima_order,
    "autosarima_full_series": future_order,
    "random_forest": {"n_estimators": 300, "max_depth": 8, "min_samples_leaf": 2, "random_state": RANDOM_SEED, "recursive": True},
    "packages": {"python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__, "matplotlib": matplotlib.__version__, "pmdarima": pmdarima.__version__, "scikit_learn": sklearn.__version__},
    "figure_validation": figure_validation,
    "limitations": [
        "Open-Meteo reanalysis/grid precipitation is not an on-site rain-gauge record.",
        "The current bounded API Auto-SARIMA search differs from the older notebook's unconstrained call; metrics are not forced to match prior claims.",
        "The storage sensitivity is a controlled model result, not field validation.",
        "No ML yield/classifier metrics are presented as real-world validation because their targets/labels are formula- or rule-derived."
    ],
}
(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

readme_lines = [
    "# WaterWallet research visualisations", "",
    f"Generated from `WaterWallet_Research_Visualisations.ipynb` on {metadata['generated_at_utc']}.", "",
    "## Successfully generated",
    "- `waterwallet_figures/fig1_monthly_rainfall_seasonality.png`",
    "- `waterwallet_figures/fig2_holdout_actual_vs_predicted.png`",
    "- `waterwallet_figures/fig3_model_metrics_comparison.png`",
    "- `waterwallet_figures/fig4_autosarima_12_month_forecast.png`",
    "- `waterwallet_figures/fig5_tank_capacity_sensitivity.png` (controlled engineering sensitivity)", "",
    "## Data and evaluation",
    "Observed daily precipitation is sourced from the documented Open-Meteo request recorded in `metadata.json`. The expected 2010-01-01 to 2025-12-31 window contains 5,844 days and aggregates to 192 months. The chronological holdout is 2022-01 through 2025-12. Metrics are calculated from the predictions saved in `holdout_actual_and_predictions_2022_2025.csv`.", "",
    "Auto-SARIMA follows the current backend service's bounded AIC search. The prior paper's benchmark used a different, unconstrained notebook call, so previous reported metrics are comparison references only—not targets.", "",
    "## Re-run",
    "Run the notebook from the repository root with a Python environment containing pandas, NumPy, Matplotlib, requests, scikit-learn, pmdarima, and the backend dependencies. It reuses the saved daily CSV when present; delete that CSV only if you intentionally want to retrieve a fresh Open-Meteo response.", "",
    "## Limitations",
    "The forecast is based on gridded/reanalysis precipitation, not a roof gauge. Forecast intervals are model-based, not guarantees. The capacity figure is a documented controlled simulation using the production daily water balance; it is not a field measurement or tank recommendation.",
]
(OUTPUT_DIR / "README.md").write_text(chr(10).join(readme_lines) + chr(10), encoding="utf-8")

zip_path = ROOT / "WaterWallet_Research_Visualisations.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    archive.write(ROOT / "WaterWallet_Research_Visualisations.ipynb", "WaterWallet_Research_Visualisations.ipynb")
    for item in OUTPUT_DIR.rglob("*"):
        if item.is_file():
            archive.write(item, item.relative_to(OUTPUT_DIR.parent))
print("Figure validation:", json.dumps(figure_validation, indent=2))
print(f"ZIP written: {zip_path}")